# Conflict Analysis (mostly superseded — see notes)

> **Heads-up:** Most of what this notebook does is now produced automatically by the
> `python -m t12_pipeline.build_rulebook.run` chain. The current rulebook
> (`db/rulebooks/mapping_rulebook_<ts>.json`) already has conflict resolution baked in
> via four buckets: `certain`, `contextual`, `no_conflict`, `unresolved`.
>
> **Where conflict data lives today:**
> - `output_docs/<ts>/build_rulebook/historical_rule_support/historical_conflicts.json` — every conflict + its bucket assignment + which `.xlsm` files contributed
> - `output_docs/<ts>/build_rulebook/historical_rule_support/auto_assigned_unresolved.json` — newly-discovered conflicts that need client review
> - The `conflict_sources_report` Excel in `data/client_review_docs/` — the client's review document
>
> The cells below were written for the old v2 rulebook shape (`{category: [descriptions]}`)
> which no longer exists. Section 1 has been rewritten to inspect the current rulebook;
> Sections 2-4 are left in place for reference but reference removed paths and will not run.

**Section 1** (updated) — count conflicts in the current rulebook and break them down by bucket.

**Sections 2-4** (legacy) — kept for reference. They reference `RULEBOOK_DB_V2_PATH`,
`CATEGORY_CLASSIFICATION_PATH`, `NEW_MAPPED_T12_DOCS_DIR` and other constants that have been
removed. They will raise `ImportError` if run as-is.

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict

# Resolve sys.path to t12_pipeline src/ so package imports work regardless of cwd.
_HERE = Path.cwd().resolve()
_SRC = next(
    (p for p in [_HERE, *_HERE.parents] if (p / 't12_pipeline' / 'config' / 'paths.py').is_file()),
    None,
)
if _SRC and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from t12_pipeline.config.paths import (
    DB_RULEBOOKS_DIR,
    MAPPED_T12_DOCS_DIR,
    OUTPUT_DOCS_BASE_DIR,
)

# The active rulebook is the newest mapping_rulebook_*.json in db/rulebooks/.
_candidates = sorted(DB_RULEBOOKS_DIR.glob('mapping_rulebook_*.json'))
if not _candidates:
    raise FileNotFoundError(
        f"No rulebook in {DB_RULEBOOKS_DIR}. "
        "Run `python -m t12_pipeline.build_rulebook.run` first."
    )
LATEST_RULEBOOK_PATH = _candidates[-1]

# Find the most recent build_rulebook output dir so we can read the conflict files.
_HRS_DIRS = sorted(OUTPUT_DOCS_BASE_DIR.glob('*/build_rulebook/historical_rule_support'))
LATEST_HRS_DIR = _HRS_DIRS[-1] if _HRS_DIRS else None

print(f'Latest rulebook:    {LATEST_RULEBOOK_PATH.name}')
print(f'Latest build outputs: {LATEST_HRS_DIR.relative_to(OUTPUT_DOCS_BASE_DIR) if LATEST_HRS_DIR else "(none yet — run build_rulebook.run)"}')

## Section 1 — Inspect conflicts in the current rulebook

Reads the latest rulebook + the latest `historical_conflicts.json` from the build chain and:
- shows the per-bucket counts (`certain` / `contextual` / `unresolved`)
- lists every conflict description and what bucket it ended up in
- highlights auto-routed conflicts (descriptions found in `.xlsm` history but not yet in the client review workbook)

In [ ]:
if LATEST_HRS_DIR is None:
    raise FileNotFoundError(
        f'No build_rulebook output yet. Run: python -m t12_pipeline.build_rulebook.run'
    )

with open(LATEST_RULEBOOK_PATH) as f:
    rulebook = json.load(f)

historical_path = LATEST_HRS_DIR / 'historical_conflicts.json'
with open(historical_path) as f:
    historical = json.load(f)

print(f'Rulebook:            {LATEST_RULEBOOK_PATH.name}')
print(f'Historical conflicts: {historical_path.relative_to(OUTPUT_DOCS_BASE_DIR)}')
print()

buckets = {
    'certain':     len(rulebook.get('certain', {})),
    'contextual':  len(rulebook.get('contextual', {})),
    'unresolved':  len(rulebook.get('unresolved', {})),
    'no_conflict': len(rulebook.get('no_conflict', {})),
}
print('Rulebook buckets:')
for k, v in buckets.items():
    print(f'  {k:<12} {v}')

print(f"\nTotal conflict descriptions (mapping_count > 1): {len(historical['conflicts'])}")
print('Review-bucket distribution:')
for k, v in historical['summary']['review_bucket_counts'].items():
    suffix = '  (informational — these were auto-routed to unresolved)' if k == 'not_in_review' else ''
    print(f'  {k:<14} {v}{suffix}')

# Auto-routed entries — descriptions found in history but not in client review.
auto_path = LATEST_HRS_DIR / 'auto_assigned_unresolved.json'
if auto_path.exists():
    with open(auto_path) as f:
        auto = json.load(f)
    print(f"\nAuto-routed conflicts (need client review): {auto['summary']['total_auto_assigned']}")
    for entry in auto['auto_assigned']:
        mappings = ' / '.join(entry['candidate_mappings'])
        print(f"  - {entry['description']:<35}  candidates: {mappings}")

In [3]:
pair_counts_for_json = defaultdict(list)
for desc_lower, entries in duplicates.items():
    cats = tuple(sorted(set(e['category'] for e in entries)))
    pair_counts_for_json[cats].append(entries[0]['original_desc'])

output_data = {
    'summary': {
        'total_categories': len(rulebook),
        'total_descriptions': sum(len(v) for v in rulebook.values()),
        'total_duplicates': len(duplicates),
    },
    'duplicates': [
        {
            'description': entries[0]['original_desc'],
            'description_normalized': desc_lower,
            'categories': [e['category'] for e in entries],
        }
        for desc_lower, entries in sorted(duplicates.items())
    ],
    'conflicts_by_category_pair': [
        {
            'category_a': pair[0],
            'category_b': pair[1] if len(pair) > 1 else pair[0],
            'conflict_count': len(descs),
            'descriptions': sorted(descs),
        }
        for pair, descs in sorted(pair_counts_for_json.items(), key=lambda x: -len(x[1]))
    ],
}

with open(OUTPUT_PATH, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f'Saved: {OUTPUT_PATH}')

Saved: /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/logic/mapping_rulebook/output_docs/2026_05_15/analysis/rulebook_duplicates_v2.json


## Section 2 — Trace Conflict Sources to Files

For every duplicate description, scans all 19 human-reviewed xlsm source files to find which
files contributed each conflicting mapping. Also flags cross-bucket conflicts (REVENUE vs EXPENSE).

**Input:** `output_docs/analysis/rulebook_duplicates.json` (v1 duplicates — has 94 conflicts)

**Output:** `output_docs/analysis/conflict_file_sources_v2.json`

In [ ]:
raise NotImplementedError(
    "Legacy — Section 2 was written for the old v2 rulebook (`{category: [descriptions]}` shape) "
    "which no longer exists. The equivalent data is now produced automatically by the build chain "
    "and lives in `output_docs/<ts>/build_rulebook/historical_rule_support/historical_conflicts.json`. "
    "See the banner at the top of this notebook."
)

# --- legacy code below for reference only ---
DUPLICATES_PATH = ANALYSIS_OUTPUT_DIR / 'rulebook_duplicates.json'
OUTPUT_JSON = ANALYSIS_OUTPUT_DIR / 'conflict_file_sources_v2.json'

with open(DUPLICATES_PATH) as f:
    dup_data = json.load(f)

dup_lookup = {
    d['description_normalized']: d['categories']
    for d in dup_data['duplicates']
}

print(f'Loaded {len(dup_lookup)} duplicate descriptions to search for.\n')

with open(CATEGORY_CLASSIFICATION_PATH) as f:
    classification = json.load(f)

def get_bucket(category):
    return classification.get(category, 'UNKNOWN')

In [5]:
xlsm_files = (
    sorted(MAPPED_T12_DOCS_DIR.glob('*.xlsm')) +
    sorted(NEW_MAPPED_T12_DOCS_DIR.glob('*.xlsm'))
)
xlsm_files = [f for f in xlsm_files if not f.name.startswith('~$')]

print(f'Scanning {len(xlsm_files)} xlsm files...\n')

desc_file_hits = defaultdict(list)

for fpath in xlsm_files:
    try:
        wb = openpyxl.load_workbook(fpath, data_only=True, read_only=True)
        if 'Financials' not in wb.sheetnames:
            wb.close()
            continue
        ws = wb['Financials']

        anchor_row = None
        for row in ws.iter_rows():
            for cell in row:
                if cell.value and isinstance(cell.value, str) and 'COMPLETE MAPPING HERE' in cell.value.upper():
                    anchor_row = cell.row
                    break
            if anchor_row:
                break

        if not anchor_row:
            wb.close()
            continue

        all_rows = list(ws.iter_rows(min_row=anchor_row + 1, values_only=True))
        wb.close()

        if not all_rows:
            continue

        headers = all_rows[0]
        try:
            desc_col = headers.index('Row Description (from T12)')
            fm_col   = headers.index('Final_Mapping')
        except ValueError:
            continue

        for data_row in all_rows[1:]:
            if not data_row or data_row[0] is None:
                break
            desc = data_row[desc_col] if desc_col < len(data_row) else None
            if not desc or not isinstance(desc, str):
                continue
            desc_norm = desc.strip().lower()
            if desc_norm not in dup_lookup:
                continue
            fm = data_row[fm_col] if fm_col < len(data_row) else None
            if not fm or not isinstance(fm, str):
                continue
            desc_file_hits[desc_norm].append({
                'file': fpath.name,
                'final_mapping': fm.strip().title(),
            })

    except Exception as e:
        print(f'  ERROR reading {fpath.name}: {e}')

Scanning 19 xlsm files...



/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


  ERROR reading 02242026_Grande Reserve at Geist_V3- VM.xlsm: name 'dup_lookup' is not defined
  ERROR reading 02252026_Willows of Coventry_V4 (1).xlsm: name 'dup_lookup' is not defined
  ERROR reading 03092026_The Flats at Arrowood_V4 (FULL BUDGET).xlsm: name 'dup_lookup' is not defined
  ERROR reading 03092026_The Tatum_V4 (assumption).xlsm: name 'dup_lookup' is not defined
  ERROR reading 03122025_District at Hurstborne_V2.xlsm: name 'dup_lookup' is not defined
  ERROR reading 260312_Melville Apartments_V18-HNW BRIDGE.xlsm: name 'dup_lookup' is not defined
  ERROR reading 031026_Mylo Apartment Homes_V2 - VM.xlsm: name 'dup_lookup' is not defined
  ERROR reading 03112026_Forma at the Park_V1.xlsm: name 'dup_lookup' is not defined
  ERROR reading 03182026_The Park at LeBlanc_V3- VM.xlsm: name 'dup_lookup' is not defined
  ERROR reading 03192026_The Jasmine Apartments_V2 - VM.xlsm: name 'dup_lookup' is not defined
  ERROR reading 03242026_Prescott Park Apartments_V1.5.xlsm: name 'dup_l

In [6]:
found_in_files = []
not_found = []

for desc_norm, conflict_cats in sorted(dup_lookup.items()):
    hits = desc_file_hits.get(desc_norm, [])

    if not hits:
        not_found.append(desc_norm)
        continue

    by_mapping = defaultdict(list)
    for h in hits:
        by_mapping[h['final_mapping']].append(h['file'])

    cat_buckets = {cat: get_bucket(cat) for cat in conflict_cats}
    unique_buckets = set(cat_buckets.values())
    cross_bucket = 'REVENUE' in unique_buckets and 'EXPENSE' in unique_buckets

    found_in_files.append({
        'description': next(
            d['description'] for d in dup_data['duplicates']
            if d['description_normalized'] == desc_norm
        ),
        'rulebook_conflict_categories': conflict_cats,
        'category_buckets': cat_buckets,
        'cross_bucket_conflict': cross_bucket,
        'file_occurrences': [
            {
                'final_mapping': mapping,
                'bucket': get_bucket(mapping),
                'files': sorted(files),
                'file_count': len(files),
            }
            for mapping, files in sorted(by_mapping.items())
        ],
        'total_files_with_this_row': len(hits),
    })

cross_bucket_entries = [e for e in found_in_files if e['cross_bucket_conflict']]

sources_output = {
    'summary': {
        'total_duplicates_in_rulebook':    len(dup_lookup),
        'duplicates_found_in_xlsm_files':  len(found_in_files),
        'cross_bucket_conflicts':          len(cross_bucket_entries),
        'duplicates_not_seen_in_any_file': len(not_found),
        'xlsm_files_scanned':             len(xlsm_files),
    },
    'conflict_sources':   found_in_files,
    'not_found_in_files': sorted(not_found),
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(sources_output, f, indent=2)

print(f'Results:')
print(f'  {len(found_in_files)} duplicate descriptions found in xlsm files')
print(f'  {len(cross_bucket_entries)} are cross-bucket (REVENUE vs EXPENSE) — most critical')
print(f'  {len(not_found)} duplicates not seen in any file (rulebook-only entries)')
print()

for entry in found_in_files:
    flag = ' CROSS-BUCKET' if entry['cross_bucket_conflict'] else ''
    print(f"  '{entry['description']}'{flag}")
    conflict_str = ' vs '.join(
        f"{c} ({entry['category_buckets'].get(c, '?')})"
        for c in entry['rulebook_conflict_categories']
    )
    print(f'    Conflict: {conflict_str}')
    for occ in entry['file_occurrences']:
        for fname in occ['files']:
            print(f"      {fname}  ->  {occ['final_mapping']} ({occ['bucket']})")
    print()

print(f'Saved: {OUTPUT_JSON}')

NameError: name 'dup_lookup' is not defined

## Section 3 — Export to Excel

Converts the conflict sources JSON to a two-tab Excel report.
- **Tab 1 — Category Conflicts:** all descriptions with multiple mappings
- **Tab 2 — Revenue vs Expense:** cross-bucket conflicts only (most critical for NOI)

**Input:** `output_docs/analysis/conflict_file_sources_v2.json` (run Section 2 first)

**Output:** `output_docs/analysis/conflict_sources_report_v2.xlsx`

In [ ]:
raise NotImplementedError(
    "Legacy — Section 3 exports the v2 conflict report to Excel. The current client review "
    "workbook is at `data/client_review_docs/conflict_sources_report_v2_J VM Notes.xlsx` and "
    "is maintained directly by the client. See the banner at the top of this notebook."
)

# --- legacy code below for reference only ---
INPUT_PATH  = ANALYSIS_OUTPUT_DIR / 'conflict_file_sources_v2.json'
OUTPUT_XLSX = ANALYSIS_OUTPUT_DIR / 'conflict_sources_report_v2.xlsx'

HEADER_FILL = PatternFill('solid', fgColor='1F4E79')
HEADER_FONT = Font(color='FFFFFF', bold=True, size=11)
DESC_FONT   = Font(bold=True, size=10)
CAT_FONT    = Font(size=10)

GROUP_FILLS = [
    PatternFill('solid', fgColor='FFFFFF'),
    PatternFill('solid', fgColor='EBF3FB'),
]

DIVIDER_BORDER = Border(bottom=Side(style='thin', color='BBBBBB'))


def write_tab(ws, entries, tab_title, include_bucket):
    ws.title = tab_title

    headers = ['Row Description', 'Category']
    if include_bucket:
        headers.append('Bucket')
    headers.append('Source Files')

    ws.append(headers)
    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 20

    shade_index = 0

    for entry in entries:
        desc = entry['description']
        fill = GROUP_FILLS[shade_index % 2]
        first_in_group = True

        for occ in entry['file_occurrences']:
            files_str = '\n'.join(occ['files'])

            row_values = [desc if first_in_group else '', occ['final_mapping']]
            if include_bucket:
                row_values.append(occ['bucket'])
            row_values.append(files_str)

            ws.append(row_values)
            r = ws.max_row

            for col in range(1, len(headers) + 1):
                cell = ws.cell(r, col)
                cell.fill = fill
                cell.alignment = Alignment(vertical='top', wrap_text=(col == len(headers)))

            ws.cell(r, 1).font = DESC_FONT if first_in_group else CAT_FONT
            ws.cell(r, 2).font = CAT_FONT

            first_in_group = False

        last_row = ws.max_row
        for col in range(1, len(headers) + 1):
            ws.cell(last_row, col).border = DIVIDER_BORDER

        shade_index += 1

    ws.column_dimensions['A'].width = 38
    ws.column_dimensions['B'].width = 30
    if include_bucket:
        ws.column_dimensions['C'].width = 10
        ws.column_dimensions['D'].width = 55
    else:
        ws.column_dimensions['C'].width = 55

    ws.freeze_panes = 'A2'

In [ ]:
with open(INPUT_PATH) as f:
    data = json.load(f)

all_entries   = data['conflict_sources']
cross_entries = [e for e in all_entries if e['cross_bucket_conflict']]

wb = openpyxl.Workbook()
write_tab(wb.active,         all_entries,   'Category Conflicts', include_bucket=False)
write_tab(wb.create_sheet(), cross_entries, 'Revenue vs Expense', include_bucket=True)

wb.save(OUTPUT_XLSX)

print(f'Saved: {OUTPUT_XLSX}')
print(f"  Tab 1 'Category Conflicts': {len(all_entries)} descriptions")
print(f"  Tab 2 'Revenue vs Expense': {len(cross_entries)} descriptions")

## Section 4 — Rulebook-First Approach Test

Tests whether looking up every row (headers, subtotals, and data rows) directly in the rulebook
is viable as an alternative to block detection.

**Result from initial testing: REJECTED — 27% header conflict rate.**

**Requires:** `output_docs/1_mapping/block_detection/` files (run `detect_blocks.py` first).

In [ ]:
raise NotImplementedError(
    "Legacy — Section 4 tested the rulebook-first row classification approach against the old "
    "v2 rulebook. That approach was rejected (27% header conflict rate) and the production "
    "pipeline now uses `detect_blocks` for classification + map_blocks for mapping. "
    "See the banner at the top of this notebook."
)

# --- legacy code below for reference only ---
with open(RULEBOOK_DB_V2_PATH) as f:
    rulebook_test = json.load(f)

reverse_map_test = {}
for category, descriptions in rulebook_test.items():
    for desc in descriptions:
        reverse_map_test[desc.strip().lower()] = category

print(f'Rulebook: {len(rulebook_test)} categories, {len(reverse_map_test)} description entries')
print(f'Rulebook path: {RULEBOOK_DB_V2_PATH}\n')

In [ ]:
json_files = sorted(BLOCK_DETECTION_DIR.glob('*.json'))
print(f'Input files ({len(json_files)} total): {BLOCK_DETECTION_DIR}')
for f in json_files:
    print(f'  - {f.name}')
print()

totals    = {'data': 0, 'header': 0, 'subtotal': 0}
found_map = {'data': 0, 'header': 0, 'subtotal': 0}
conflicts = []

for jfile in json_files:
    with open(jfile) as f:
        data = json.load(f)

    fname = data['file']
    fmt   = data['format']

    for row in data['rows']:
        row_type = row['type']
        desc     = row['description'].strip()
        mapping  = reverse_map_test.get(desc.lower())

        totals[row_type] = totals.get(row_type, 0) + 1

        if mapping:
            found_map[row_type] = found_map.get(row_type, 0) + 1
            if row_type in ('header', 'subtotal'):
                conflicts.append({
                    'file':             fname,
                    'format':           fmt,
                    'row_number':       row['row'],
                    'row_type':         row_type,
                    'description':      desc,
                    'rulebook_maps_to': mapping,
                    'section_context':  row.get('section_context', ''),
                })

print('=' * 65)
print('SUMMARY — Rulebook lookup across ALL row types')
print('=' * 65)
for t in ['data', 'header', 'subtotal']:
    total = totals.get(t, 0)
    hit   = found_map.get(t, 0)
    miss  = total - hit
    pct   = (hit / total * 100) if total else 0
    print(f'  {t.upper():<10}  total={total:4}  found={hit:4}  not_found={miss:4}  ({pct:.0f}% hit rate)')

print()
print(f'CONFLICTS (header/subtotal rows that rulebook can map): {len(conflicts)}')
print('These rows would be INCORRECTLY treated as DATA rows under rulebook-first logic.')

if conflicts:
    print()
    print('-' * 65)
    by_file = defaultdict(list)
    for c in conflicts:
        by_file[c['file']].append(c)

    for fname, rows in sorted(by_file.items()):
        print(f"\nFile: {fname}  (format: {rows[0]['format']})")
        for c in rows:
            rn = c['row_number']
            rt = c['row_type'].upper()
            d  = c['description']
            m  = c['rulebook_maps_to']
            print(f'  row {rn:3d}  [{rt:<8}]  {d}  -->  rulebook: {m}')

print()
print('=' * 65)
print('VERDICT')
print('=' * 65)

header_found   = found_map.get('header', 0)
header_total   = totals.get('header', 1)
subtotal_found = found_map.get('subtotal', 0)
subtotal_total = totals.get('subtotal', 1)

header_conflict_pct   = (header_found / header_total) * 100
subtotal_conflict_pct = (subtotal_found / subtotal_total) * 100

print(f'  HEADER   conflict rate: {header_conflict_pct:.0f}%  ({header_found}/{header_total} rows)')
print(f'  SUBTOTAL conflict rate: {subtotal_conflict_pct:.0f}%  ({subtotal_found}/{subtotal_total} rows)')
print()

if header_conflict_pct > 15:
    print('  HIGH header conflict rate — rulebook-first for HEADER rows is NOT safe.')
    print('  Recommendation: keep current detect_blocks classification for headers.')
else:
    print('  Low conflict rate — rulebook-first may be viable.')

if subtotal_conflict_pct > 10:
    print('  HIGH subtotal conflict rate — rulebook-first for SUBTOTAL rows is NOT safe.')
else:
    print('  Low subtotal conflict rate — rulebook-first may be viable for subtotals.')